# Python Excel Automation: การกรองข้อมูลด้วย OpenPyXl - ตอนที่ 3

**คำอธิบาย:** การกรองข้อมูลขั้นสูงด้วย OpenPyXL สำหรับ Excel

**หัวข้อที่ครอบคลุม:**
- สำรวจขนาดของ Spreadsheet
- เก็บแถวหัวตาราง
- การใช้งาน Filter และทำความเข้าใจกระบวนการ
- การคัดลอกข้อมูลที่กรองแล้ว
- การใช้หลายเงื่อนไขในการกรอง
- การกรองตามชื่อลูกค้าเฉพาะ
- การกรองวันที่ใน Excel
- ตัวอย่างการใช้งานจริง

In [ ]:
# 📦 ติดตั้ง packages ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import openpyxl
    print("✅ ติดตั้ง packages ทั้งหมดแล้ว")
except ImportError:
    %pip install openpyxl

In [ ]:
# 📚 นำเข้าส่วนประกอบ openpyxl
# load_workbook — เปิดไฟล์ .xlsx ที่มีอยู่
# Workbook — สร้างไฟล์ .xlsx ใหม่ตั้งแต่ต้น
from openpyxl import load_workbook, Workbook
import os

## เตรียมข้อมูล: สร้างข้อมูลตัวอย่าง

In [ ]:
# 📝 สร้างข้อมูลตัวอย่าง — ชุดข้อมูล "Superstore" ย่อสำหรับฝึกกรองข้อมูล
# ws.append() เพิ่มแต่ละแถวที่ด้านล่างของ sheet
# ข้อมูลมีหลายหมวดหมู่ รัฐ วันที่ สำหรับฝึกกรอง
wb_sample = Workbook()
ws = wb_sample.active
ws.title = 'Orders'

headers = ['Order ID', 'Customer', 'Product', 'Category', 'Sales', 'Quantity', 'State', 'Order Date', 'Ship Mode']
ws.append(headers)

sample_data = [
    [1001, 'John Smith', 'Widget A', 'Technology', 250.50, 3, 'California', '2024-01-15', 'Standard'],
    [1002, 'Jane Doe', 'Widget B', 'Furniture', 1230.00, 1, 'Texas', '2024-02-20', 'Express'],
    [1003, 'John Smith', 'Widget C', 'Office Supplies', 89.99, 5, 'New York', '2024-03-10', 'Standard'],
    [1004, 'Alice Brown', 'Widget A', 'Technology', 450.50, 2, 'California', '2024-04-05', 'Standard'],
    [1005, 'Bob Wilson', 'Widget D', 'Furniture', 3475.00, 1, 'Florida', '2024-05-18', 'Express'],
    [1006, 'Jane Doe', 'Widget E', 'Technology', 899.00, 4, 'Texas', '2024-06-22', 'Same Day'],
    [1007, 'John Smith', 'Widget F', 'Office Supplies', 45.99, 10, 'New York', '2024-07-30', 'Standard'],
    [1008, 'Alice Brown', 'Widget G', 'Technology', 1250.00, 1, 'California', '2024-08-12', 'Express'],
    [1009, 'Charlie Lee', 'Widget H', 'Furniture', 675.00, 2, 'Florida', '2024-09-05', 'Standard'],
    [1010, 'Jane Doe', 'Widget I', 'Office Supplies', 120.00, 8, 'Texas', '2024-10-15', 'Express'],
]

for row in sample_data:
    ws.append(row)

wb_sample.save('Superstore.xlsx')
print("✅ สร้าง Superstore.xlsx ตัวอย่างแล้ว")

## 1. โหลด Workbook และสำรวจขนาดข้อมูล

In [ ]:
# 📂 โหลด workbook แล้วสำรวจขนาดข้อมูล
# .dimensions = ช่วงที่ใช้งาน เช่น "A1:I11"
# .max_row / .max_column = จำนวนแถวและคอลัมน์ที่มีข้อมูล
wkb = load_workbook('Superstore.xlsx')
print(f"Sheet ทั้งหมด: {wkb.sheetnames}")

order = wkb['Orders']
print(f"ขนาดข้อมูล: {order.dimensions}")
print(f"จำนวนแถว: {order.max_row}")
print(f"จำนวนคอลัมน์: {order.max_column}")

## 2. เก็บแถวหัวตาราง (Header)

In [ ]:
# 📋 เก็บแถวหัวตาราง — จะใช้หาตำแหน่งคอลัมน์สำหรับกรองข้อมูล
# order[1] = แถวที่ 1 → list comprehension ดึงค่าของแต่ละเซลล์
# header.index('ชื่อคอลัมน์') ให้ตำแหน่งคอลัมน์ (เริ่มจาก 0)
header = [cell.value for cell in order[1]]
print(f"หัวตาราง: {header}")
print(f"จำนวนคอลัมน์: {len(header)}")

## 3. ตั้งค่า AutoFilter

In [ ]:
# 🔽 ตั้งค่า AutoFilter — เพิ่มลูกศร dropdown ตัวกรองใน Excel
# .auto_filter.ref = ช่วงที่จะมี dropdown
# ⚠️ นี่แค่เพิ่ม UI dropdown เท่านั้น — openpyxl ไม่ได้ซ่อนแถวจริง!
# เราต้องกรองข้อมูลเองด้วย Python (แสดงในเซลล์ถัดไป)
order.auto_filter.ref = order.dimensions
print(f"ช่วง AutoFilter: {order.auto_filter.ref}")

## 4. กรองข้อมูลและคัดลอกไป Sheet ใหม่
เนื่องจาก openpyxl ไม่ได้กรองแถวจริง (แค่ตั้งค่า metadata ตัวกรอง) เราจึงกรองข้อมูลด้วย Python เอง

In [ ]:
# 🔍 กรองเงื่อนไขเดียว — คัดลอกแถวที่ตรงเงื่อนไขไป sheet ใหม่
# ขั้นตอน:
#   1. สร้าง sheet ใหม่สำหรับผลลัพธ์
#   2. คัดลอกแถวหัวตาราง
#   3. วนลูปแถวข้อมูล ตรวจเงื่อนไข คัดลอกแถวที่ตรง
# header.index('Category') หาตำแหน่งคอลัมน์แบบอัตโนมัติ
filter_worksheet = wkb.create_sheet('Filtered_Data')

# คัดลอกหัวตารางไป sheet ใหม่
for col_idx, cell in enumerate(order[1], 1):
    filter_worksheet.cell(row=1, column=col_idx, value=cell.value)

# กรอง: Category = 'Technology'
filter_column = 'Category'
filter_value = 'Technology'
col_index = header.index(filter_column)

dest_row = 2
for row in order.iter_rows(min_row=2, values_only=False):
    if row[col_index].value == filter_value:
        for col_idx, cell in enumerate(row, 1):
            filter_worksheet.cell(row=dest_row, column=col_idx, value=cell.value)
        dest_row += 1

print(f"✅ กรองได้ {dest_row - 2} แถว ที่ {filter_column} = '{filter_value}'")

## 5. กรองหลายเงื่อนไข (AND)

In [ ]:
# 🔍 หลายเงื่อนไข (AND) — ทั้งสองเงื่อนไขต้องเป็นจริง
# ใช้ `and` ของ Python เพื่อรวมเงื่อนไข
# ตัวอย่าง: Category='Technology' และ State='California'
multi_filter_ws = wkb.create_sheet('Multi_Filter')

for col_idx, cell in enumerate(order[1], 1):
    multi_filter_ws.cell(row=1, column=col_idx, value=cell.value)

cat_idx = header.index('Category')
state_idx = header.index('State')

dest_row = 2
for row in order.iter_rows(min_row=2, values_only=False):
    if row[cat_idx].value == 'Technology' and row[state_idx].value == 'California':
        for col_idx, cell in enumerate(row, 1):
            multi_filter_ws.cell(row=dest_row, column=col_idx, value=cell.value)
        dest_row += 1

print(f"✅ กรองได้ {dest_row - 2} แถว (Technology + California)")

## 6. กรองตามชื่อลูกค้าเฉพาะ

In [ ]:
# 👤 กรองตามชื่อลูกค้า — หาคำสั่งซื้อทั้งหมดของลูกค้าที่ระบุ
# รูปแบบเดิม: สร้าง sheet → คัดลอกหัวตาราง → วนลูป & เทียบค่า → คัดลอกแถว
customer_ws = wkb.create_sheet('Customer_Filter')

for col_idx, cell in enumerate(order[1], 1):
    customer_ws.cell(row=1, column=col_idx, value=cell.value)

customer_idx = header.index('Customer')
target_customer = 'John Smith'

dest_row = 2
for row in order.iter_rows(min_row=2, values_only=False):
    if row[customer_idx].value == target_customer:
        for col_idx, cell in enumerate(row, 1):
            customer_ws.cell(row=dest_row, column=col_idx, value=cell.value)
        dest_row += 1

print(f"✅ พบ {dest_row - 2} คำสั่งซื้อของ '{target_customer}'")

## 7. กรองตามช่วงวันที่

In [ ]:
# 📅 กรองตามช่วงวันที่ — หาคำสั่งซื้อในช่วงวันที่ที่กำหนด
# แปลงเป็น string เพื่อเปรียบเทียบ (ใช้ได้เมื่อวันที่เก็บเป็น 'YYYY-MM-DD')
# การเปรียบเทียบ string ใช้ได้กับรูปแบบ ISO (เรียงตามตัวอักษร = เรียงตามเวลา)
date_ws = wkb.create_sheet('Date_Filter')

for col_idx, cell in enumerate(order[1], 1):
    date_ws.cell(row=1, column=col_idx, value=cell.value)

date_idx = header.index('Order Date')
start_date = '2024-03-01'
end_date = '2024-08-31'

dest_row = 2
for row in order.iter_rows(min_row=2, values_only=False):
    date_val = str(row[date_idx].value)
    if start_date <= date_val <= end_date:
        for col_idx, cell in enumerate(row, 1):
            date_ws.cell(row=dest_row, column=col_idx, value=cell.value)
        dest_row += 1

print(f"✅ พบ {dest_row - 2} คำสั่งซื้อ ระหว่าง {start_date} ถึง {end_date}")

## 8. บันทึกผลลัพธ์

In [ ]:
# 💾 บันทึกและปิด — sheet ที่กรองแล้วทั้งหมดอยู่ใน workbook
wkb.save('openpyxl_filtered_data.xlsx')
print(f"✅ บันทึกแล้ว พร้อม sheet: {wkb.sheetnames}")
wkb.close()
print("✅ ปิด Workbook แล้ว")

---

## 🎮 Mini Project: ค้นหาข้อมูลลูกค้า

ทดสอบการกรองข้อมูล Excel ที่เรียนมาใน Notebook นี้!

> 💡 **คำตอบ:** ดูไฟล์ `answers/04_answers.ipynb`

---

### โจทย์ที่ 1: กรองข้อมูลตามเงื่อนไขเดียว 🟢
ใช้ไฟล์ `Random Sales Data.xlsx`:
1. เปิดไฟล์ด้วย `load_workbook()`
2. หา header ทั้งหมดและ print ออกมา
3. กรองเฉพาะแถวที่ **Region = "East"**
4. สร้าง sheet ใหม่ชื่อ `"East_Sales"` แล้วใส่ข้อมูลที่กรอง
5. Print จำนวนแถวที่กรองได้
6. บันทึกไฟล์

> 💡 Hint: ใช้ `header.index('Region')` หาตำแหน่งคอลัมน์

In [ ]:
# โจทย์ที่ 1: กรองข้อมูลตามเงื่อนไขเดียว
# เขียนโค้ดของคุณที่นี่ 👇


### โจทย์ที่ 2: กรองหลายเงื่อนไข (AND) 🟡
ใช้ไฟล์ `Random Sales Data.xlsx`:
1. กรองข้อมูลที่ **Region = "West"** AND **Price > 50**
2. สร้าง sheet ชื่อ `"West_Expensive"` 
3. Copy header + ข้อมูลที่กรองไปยัง sheet ใหม่
4. Print จำนวนแถวที่กรองได้

> 💡 Hint: ใช้ 2 เงื่อนไข `if row[region_idx] == "West" and row[price_idx] > 50`

In [ ]:
# โจทย์ที่ 2: กรองหลายเงื่อนไข
# เขียนโค้ดของคุณที่นี่ 👇


### โจทย์ที่ 3: สร้าง Sheet แยกตาม Region อัตโนมัติ 🔴
เขียนโค้ดที่:
1. อ่าน unique values ของคอลัมน์ `Region`
2. Loop สร้าง sheet ใหม่สำหรับแต่ละ Region
3. Copy header + ข้อมูลที่ตรงกับ Region นั้นไปยัง sheet
4. Print สรุป: ชื่อ Region และจำนวนแถวใน sheet

> 💡 Hint: ใช้ `set()` หาค่า unique + loop สร้าง sheet

In [ ]:
# โจทย์ที่ 3: สร้าง Sheet แยกตาม Region อัตโนมัติ
# เขียนโค้ดของคุณที่นี่ 👇
